# M3 rules-only matching reference harness

Python reference harness + C# gate spec. No database, network, cloud model, or LLM for decisions. Run top to bottom.


In [12]:
import requests

response = requests.get(
    f"{API_URL}/health",
    timeout=20
)

print(response.status_code)
print(response.json())

200
{'status': 'online'}


In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\mohan\PycharmProjects\SyntheticData"
)

# Add project folder to Python import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import Kaggle API client
from llm.kaggle_client import ask_llm, kaggle_health

print("Kaggle client imported successfully!")

Kaggle client imported successfully!


In [14]:
import os
from getpass import getpass

# Your CURRENT Kaggle Cloudflare URL
os.environ["KAGGLE_LLM_API_URL"] = (
    "https://purchases-ceo-upc-platinum.trycloudflare.com"
)

# Enter the same token stored in Kaggle Secrets
os.environ["KAGGLE_LLM_API_TOKEN"] = getpass(
    "Enter SYNTHETICDATA_API_TOKEN: "
)

print("Kaggle API configuration loaded.")

Kaggle API configuration loaded.


In [15]:
from llm.kaggle_client import kaggle_health

health = kaggle_health()

print(health)

{'status': 'online'}


In [16]:
from llm.kaggle_client import ask_llm

answer = ask_llm([
    {
        "role": "user",
        "content": "Reply with exactly: KAGGLE_LLM_OK"
    }
])

print(answer)

We need answer exactly KAGGLE_LLM_OK. No extra.
</think>

KAGGLE_LLM_OK


## 0. Header + config


### Optional Kaggle inference diagnostics

الـ Kaggle client ده diagnostic/backend adapter فقط. مش بيتستخدم في gate، ranking، feasibility، batch، fairness، أو تفسير النتائج. مفيش اتصال تلقائي عند تشغيل Run All؛ شغّل خلايا الاختبار يدويًا بعد ضبط متغيرات البيئة. الرسائل تفضل role/content objects ومفيش ملفات المشروع بتتبعت تلقائيًا.


In [17]:
import os
from llm.kaggle_client import ask_llm, kaggle_health

KAGGLE_LLM_API_URL = os.getenv("KAGGLE_LLM_API_URL")
KAGGLE_LLM_CONFIGURED = bool(KAGGLE_LLM_API_URL and os.getenv("KAGGLE_LLM_API_TOKEN"))
print("Kaggle LLM configured:", KAGGLE_LLM_CONFIGURED)


Kaggle LLM configured: True


In [18]:
# Optional live health check. Uncomment and run manually after setting both environment variables.
# health = kaggle_health()
# print(health)


In [19]:
# Optional minimal model check. This is diagnostic only and sends only the literal test message below.
# answer = ask_llm([{"role": "user", "content": "Reply with exactly: KAGGLE_LLM_OK"}])
# print(answer)
# assert "KAGGLE_LLM_OK" in answer


In [20]:
from __future__ import annotations
import hashlib,hmac,json,math,os,re,unicodedata
from collections import Counter
from dataclasses import asdict,dataclass
from datetime import date,timedelta
from pathlib import Path
from typing import Any,Literal
import numpy as np
import pandas as pd
import yaml
from pydantic import BaseModel,ConfigDict,Field
from jsonschema import Draft7Validator,Draft202012Validator
TODAY=date(2026,11,2); RANDOM_SEED=42; PIPELINE_VERSION="m3-rules-only-reference/1.0.0"; np.random.seed(RANDOM_SEED)
@dataclass(frozen=True)
class PolicyConfig:
 candidate_dir:str="Final Data/candidates"; job_dir:str="Final Data/jobs"; registry_path:str="registry/registry.yaml"; manifest_path:str="out/manifest.json"; ground_truth_path:str="out/ground_truth.json"; candidate_schema_path:str="schemas/candidate_v0_2.json"; job_schema_path:str="schemas/job_m2.json"; audit_store_dir:str="audit_store"; output_dir:str="outputs"; d02_sustained_threshold_min:int=15; d02_review_on_frequent:bool=True; d02_lower_confidence:bool=False; use_of03_for_d05:bool=False; use_d06_rule:bool=False; d06_use_max_of01_pc24:bool=True; allow_d08_elevator:bool=True; allow_d08_stairs:bool=True; accommodation_catalog_enabled:bool=True; borderline_fraction:float=.05; too_many_unknown_essential:int=2; stale_months:int=12; confidence_medium_unknown_fraction:float=.20; confidence_low_unknown_fraction:float=.50; bootstrap_replicates:int=1000; bootstrap_seed:int=42; fairness_di_low:float=.80; fairness_di_high:float=1.25; privacy_blocklist:tuple=("capability_value","diagnosis","disability","score","WH_D")
CONFIG=PolicyConfig(); MAPPING_VERSION="mapping-v0"
def canonical_json(x:Any)->str:
 def norm(v):
  if isinstance(v,dict):return {unicodedata.normalize("NFC",str(k)):norm(z) for k,z in sorted(v.items(),key=lambda p:str(p[0]))}
  if isinstance(v,list):
   a=[norm(z) for z in v]
   if a and all(isinstance(z,str) for z in a):return sorted(a)
   if a and all(isinstance(z,dict) and "name" in z for z in a):return sorted(a,key=lambda z:z["name"])
   if a and all(isinstance(z,dict) and ("item_code" in z or "dimension_code" in z) for z in a):
    k="item_code" if "item_code" in a[0] else "dimension_code";return sorted(a,key=lambda z:z[k])
   return a
  return unicodedata.normalize("NFC",v) if isinstance(v,str) else v
 return json.dumps(norm(x),ensure_ascii=False,sort_keys=True,separators=(",",":"),allow_nan=False)
def safe_log(event:str,fields:dict|None=None)->dict:return {"event":re.sub(r"[^\w.-]","_",event),"fields":fields or {}}
def read_json(p,default):return json.loads(Path(p).read_text(encoding="utf-8")) if Path(p).exists() else default
def golden_candidate_fixture():return {"schema_version":"0.2","candidate_id":"cand_0001","job_category":"mobility","disability":"AUDIT_ONLY","skills":[],"cv_text":{},"capabilities":[{"item_code":"PC01","value":38,"unit":"kg","method":"self_reported"},{"item_code":"PC02","value":21,"unit":"kg","method":"self_reported"},{"item_code":"PC03","value":30,"unit":"kg","method":"self_reported"},{"item_code":"PC04","value":31,"unit":"kg","method":"self_reported"},{"item_code":"PC05","value":175,"unit":"m","method":"self_reported"},{"item_code":"PC06","value":170,"unit":"kg","method":"self_reported"},{"item_code":"PC07","value":180,"unit":"cm","method":"self_reported"},{"item_code":"PC11","value":1060,"unit":"m","method":"self_reported"},{"item_code":"PC12","value":7,"unit":"hours","method":"self_reported"},{"item_code":"PC13","value":7,"unit":"flights","method":"self_reported"},{"item_code":"PC14","value":21,"unit":"cm","method":"self_reported"},{"item_code":"PC15","value":11,"unit":"1:x","method":"self_reported"},{"item_code":"PC24","value":121,"unit":"minutes","method":"self_reported"},{"item_code":"WH_D1_2","score":None}],"input_methods":["keyboard","voice","touch"],"usable_formats":["audio"],"communication_channels":["spoken","written"],"at_needs":["screen_reader"],"preferences":{"remote":False,"location":"Cairo"}}
def golden_job_fixture():return {"job_id":"job_0001","job_version":1,"status":"published","job_type":"mobility","title":"Installation Electrician","work_mode":"hybrid","location":"Alexandria","demands":[{"dimension_code":"D01","value_state":"known","value":20,"unit":"kg","frequency_band":"never","essential":False},{"dimension_code":"D02","value_state":"known","value":120,"unit":"kg","frequency_band":"frequent","duration_min":28,"essential":True},{"dimension_code":"D05","value_state":"known","value":173,"unit":"min","essential":False},{"dimension_code":"D06","value_state":"unknown","value":None,"unit":"min","essential":False},{"dimension_code":"D07","value_state":"known","value":610,"unit":"m","essential":False},{"dimension_code":"D08","value_state":"known","value":True,"essential":True},{"dimension_code":"D09","value_state":"known","value":1150,"unit":"mm","essential":False},{"dimension_code":"D10","value_state":"known","value":1450,"unit":"mm","essential":False},{"dimension_code":"D11","value_state":"known","value":{"typical":79.8,"peak":92.2},"unit":"dB(A)","essential":False},{"dimension_code":"D12","value_state":"known","value":450,"unit":"lux","essential":False},{"dimension_code":"D13","value_state":"known","value":{"lowest":21.3,"highest":35.8},"unit":"°C","essential":False},{"dimension_code":"D15","value_state":"known","value":["channel:spoken","channel:written","provision:captioning"],"essential":False},{"dimension_code":"D16","value_state":"known","value":["stepwise","verbal","written"],"essential":False},{"dimension_code":"D17","value_state":"known","value":["hybrid"],"essential":False}],"workplace":[{"item_code":"E09","value_state":"known","value":True,"verification_level":"third_party_audited","measured_at":"2024-10-16"},{"item_code":"E10","value_state":"known","value":True,"verification_level":"self_reported","measured_at":"2025-08-20"},{"item_code":"E11","value_state":"known","value":85,"unit":"m","verification_level":"self_reported","measured_at":"2024-05-01"},{"item_code":"E12","value_state":"unknown","value":None,"measured_at":None}]}
CANDIDATE_SCHEMA_FALLBACK={"$schema":"http://json-schema.org/draft-07/schema#","type":"object","additionalProperties":False,"required":["schema_version","candidate_id","job_category","disability","skills","cv_text","capabilities","input_methods","usable_formats","communication_channels","at_needs","preferences"],"properties":{"schema_version":{"const":"0.2"},"candidate_id":{"type":"string"},"job_category":{"enum":["mobility","office","both"]},"disability":{"type":"string"},"skills":{"type":"array"},"cv_text":{"type":"object"},"capabilities":{"type":"array"},"input_methods":{"type":"array"},"usable_formats":{"type":"array"},"communication_channels":{"type":"array"},"at_needs":{"type":"array"},"preferences":{"type":"object"}}}
class Capability(BaseModel):
 model_config=ConfigDict(extra="forbid")
 item_code:str; value:float|None=None; score:int|None=None; unit:str|None=None; method:str|None=None; value_state:Literal["known","unknown"]; used_by_gate:bool=True
class InternalCandidate(BaseModel):
 model_config=ConfigDict(extra="forbid")
 schema_version:Literal["0.2"]; candidate_id:str; job_category:Literal["mobility","office","both"]; skills:list[dict]=Field(default_factory=list); cv_text:dict[str,str|None]=Field(default_factory=dict); capabilities:list[Capability]; input_methods:list[str]; usable_formats:list[str]; communication_channels:list[str]; at_needs:list[str]; preferences:dict
def privacy_border(raw:dict,schema:dict,audit_dir:Path)->tuple[dict,dict]:
 """Validate raw, save diagnosis solely in isolated audit file, and strip it before gate."""
 Draft7Validator(schema).validate(raw); audit={"candidate_id":raw["candidate_id"],"disability":raw["disability"]};audit_dir.mkdir(parents=True,exist_ok=True);(audit_dir/(raw["candidate_id"]+".json")).write_text(json.dumps(audit,ensure_ascii=False),encoding="utf-8")
 clean={k:v for k,v in raw.items() if k!="disability"};caps=[]
 for x in clean["capabilities"]:
  code=x["item_code"]
  if code.startswith("WH_D6_"):continue
  ignored=bool(re.fullmatch(r"H[1-3]",code)); known=not ignored and x.get("value",None) is not None or (not ignored and x.get("score",None) is not None)
  caps.append({**x,"value_state":"known" if known else "unknown","used_by_gate":not ignored})
 clean["capabilities"]=caps;out=InternalCandidate.model_validate(clean).model_dump()
 assert "disability" not in out
 return out,audit
def load_inputs(cfg=CONFIG):
 cs=[json.loads(p.read_text(encoding="utf-8")) for p in sorted(Path(cfg.candidate_dir).glob("*.json"))] if Path(cfg.candidate_dir).exists() else []
 js=[json.loads(p.read_text(encoding="utf-8")) for p in sorted(Path(cfg.job_dir).glob("*.json"))] if Path(cfg.job_dir).exists() else []
 reg=yaml.safe_load(Path(cfg.registry_path).read_text(encoding="utf-8")) if Path(cfg.registry_path).exists() else {}
 man=read_json(cfg.manifest_path,{"seed":42,"count":0})
 csch=read_json(cfg.candidate_schema_path,CANDIDATE_SCHEMA_FALLBACK)
 jsch=read_json(cfg.job_schema_path,{"type":"object","required":["job_id","job_version","status","job_type","demands"]})
 return cs or [golden_candidate_fixture()],js or [golden_job_fixture()],reg,man,csch,jsch
def normalize_quantity(v,unit,target):
 if v is None:return None
 u=(unit or "").strip().lower();t=(target or "").strip().lower()
 if u==t or (u,t) in {("min","minutes"),("minutes","min")}:return float(v)
 f={("hours","min"):60,("hour","min"):60,("h","min"):60,("seconds","min"):1/60,("seconds","minutes"):1/60,("cm","mm"):10,("m","mm"):1000,("hours","minutes"):60}.get((u,t))
 return float(v)*f if f is not None else None
def capmap(c):return {x["item_code"]:x for x in c["capabilities"]}
def capval(c,code,reg):
 x=capmap(c).get(code)
 if not x or x["value_state"]!="known" or x.get("value") is None:return None
 meta=reg.get("candidate_items",{}).get(code,{})
 return normalize_quantity(x["value"],x.get("unit",meta.get("unit")),meta.get("unit"))
def findd(j,code):return next((x for x in j.get("demands",[]) if x["dimension_code"]==code),None)
def findw(j,code):return next((x for x in j.get("workplace",[]) if x["item_code"]==code),None)
MAPPING={"D01":{"status":"AMBIGUOUS","items":["PC01","PC02","PC03","PC04","PC19"]},"D02":{"status":"CLEAR","items":["PC06"]},"D03":{"status":"CLEAR","items":["input_methods"]},"D04":{"status":"CLEAR","items":["OF01"]},"D05":{"status":"CLEAR","items":["PC24"]},"D06":{"status":"AMBIGUOUS","items":["OF01","PC24"]},"D07":{"status":"CLEAR","items":["PC11"]},"D08":{"status":"AMBIGUOUS","items":["PC13","E09"]},"D09":{"status":"NO_MATCH"},"D10":{"status":"NO_MATCH"},"D11":{"status":"NO_MATCH"},"D12":{"status":"NO_MATCH"},"D13":{"status":"NO_MATCH"},"D14":{"status":"CLEAR","items":["usable_formats"]},"D15":{"status":"CLEAR","items":["communication_channels"]},"D16":{"status":"AMBIGUOUS"},"D17":{"status":"CLEAR","items":["preferences.remote"]}}
ACCOMMODATION_CATALOG=[{"need_or_gap":"visual_to_audio","resolution_id":"screen_reader_audio","resolves_dimension":"D14","candidate_need":"screen_reader"},{"need_or_gap":"communication","resolution_id":"captioning","resolves_dimension":"D15","candidate_need":"captioning"},{"need_or_gap":"input_method","resolution_id":"alternative_input","resolves_dimension":"D03","candidate_need":"switch_or_gaze"}]
@dataclass(frozen=True)
class Comparison:
 status:str; reason_code:str; detail:dict; resolution:str|None=None
def R(s,r,**detail):return Comparison(s,r,detail)
def catalog(code,need=None):
 if not CONFIG.accommodation_catalog_enabled:return None
 for x in ACCOMMODATION_CATALOG:
  if x["resolves_dimension"]==code and (need is None or need in (x["need_or_gap"],x.get("candidate_need"))):return x["resolution_id"]
 return None
def compare(code,c,j,reg,cfg=CONFIG)->Comparison:
 """Pure mapping comparator; incomplete evidence remains Unknown."""
 d=findd(j,code)
 if d is None:return R("Unknown","demand_missing")
 if d.get("value_state")!="known" or d.get("value") is None:return R("Unknown","demand_unknown")
 caps=capmap(c);v=d["value"]
 if code=="D01":
  band=d.get("frequency_band"); items=("PC01","PC04") if band=="occasional" else (("PC02",) if band in ("frequent","constant") else ("PC01","PC02","PC03","PC04","PC19"))
  vals=[capval(c,x,reg) for x in items];vals=[x for x in vals if x is not None]
  return R("Unknown","candidate_value_missing") if not vals else R("Met" if max(vals)>=v else "Gap","compared_capacity",compared=True)
 simple={"D02":("PC06","kg"),"D04":("OF01","minutes"),"D05":("PC24","minutes"),"D07":("PC11","m")}
 if code in simple:
  item,target=simple[code]; choices=[item]
  if code=="D05" and cfg.use_of03_for_d05:choices=["PC24","OF03"]
  pairs=[]
  for x in choices:
   cv=capval(c,x,reg);unit=reg.get("candidate_items",{}).get(x,{}).get("unit",target);dv=normalize_quantity(v,d.get("unit"),unit)
   if cv is not None and dv is not None:pairs.append((cv,dv))
  return R("Unknown","candidate_value_missing") if not pairs else R("Met" if any(a>=b for a,b in pairs) else ("Unknown" if len(pairs)<len(choices) else "Gap"),"compared_capacity",compared=True)
 if code=="D03":
  if not c["input_methods"]:return R("Unknown","candidate_value_missing")
  if set(c["input_methods"])&set(v):return R("Met","overlap",compared=True)
  z=catalog(code,"input_method");return Comparison("Met with accommodation","catalog_resolution",{"compared":True},z) if z else R("Gap","no_overlap",compared=True)
 if code=="D08":
  if v is True:return R("Met","step_free_route",compared=True)
  e=findw(j,"E09")
  if cfg.allow_d08_elevator and e and e.get("value_state")=="known" and e.get("value") is True:return Comparison("Met with accommodation","workplace_elevator",{"compared":True},"E09_elevator")
  stairs=capval(c,"PC13",reg)
  if stairs is None:return R("Unknown","candidate_value_missing")
  return R("Met","stairs_capacity",compared=True) if cfg.allow_d08_stairs and stairs>=1 else R("Gap","route_steps_no_resolution",compared=True)
 if code=="D14":
  if set(c["usable_formats"])&set(v):return R("Met","overlap",compared=True)
  z=catalog(code,"screen_reader");return Comparison("Met with accommodation","catalog_resolution",{"compared":True},z) if z else R("Gap","no_overlap",compared=True)
 if code=="D15":
  channels={x.split(":",1)[1] for x in v if x.startswith("channel:")}
  if set(c["communication_channels"])&channels:return R("Met","overlap",compared=True)
  for p in sorted(x.split(":",1)[1] for x in v if x.startswith("provision:")):
   z=catalog(code,p)
   if z:return Comparison("Met with accommodation","catalog_resolution",{"compared":True},z)
  return R("Gap","no_overlap",compared=True)
 if code=="D06":
  if not cfg.use_d06_rule:return R("Unknown","mapping_unconfirmed")
  vals=[capval(c,x,reg) for x in ("OF01","PC24")];vals=[x for x in vals if x is not None]
  return R("Unknown","candidate_value_missing") if not vals else R("Met" if max(vals)>=v else "Gap","compared_capacity",compared=True)
 if code=="D16":return R("Unknown","mapping_unconfirmed")
 if code in ("D09","D10","D11","D12","D13"):return R("Unknown","no_candidate_counterpart")
 if code=="D17":
  options=set(v);remote=bool(c["preferences"].get("remote"));mismatch=("remote" not in options if remote else options!={"remote"})
  return R("Met","preference_mismatch" if mismatch else "preference_match",informational=True,preference_status="preference_mismatch" if mismatch else "preference_match")
 return R("Unknown","mapping_unconfirmed")
def stale_lists(j,cfg=CONFIG):
 cutoff=date(TODAY.year-cfg.stale_months,TODAY.month,TODAY.day);stale=[];undated=[]
 for x in j.get("workplace",[]):
  try:dt=date.fromisoformat(x["measured_at"]) if x.get("measured_at") else None
  except ValueError:dt=None
  if dt is None:undated.append(x["item_code"])
  elif dt<cutoff:stale.append(x["item_code"])
 return sorted(stale),sorted(undated)
def sanity(c,reg):
 flags=[]
 for a,b in reg.get("constraints_le",[]):
  x,y=capval(c,a,reg),capval(c,b,reg)
  if x is not None and y is not None and x>y:flags.append(f"sanity:{a}_gt_{b}")
 return flags
def config_hash():return hashlib.sha256(canonical_json(asdict(CONFIG)).encode()).hexdigest()[:16]
OUTPUT_SCHEMA={"$schema":"https://json-schema.org/draft/2020-12/schema","type":"object","required":["candidate_id","job_id","job_version","dimension_results","feasibility","final_binary","unverified_flag","essential_unknown_count","similarity","confidence","missing_info","review_flags","template_data","employer_category","model_version","input_hash"],"properties":{"candidate_id":{"type":"string"},"job_id":{"type":"string"},"job_version":{"type":"integer"},"dimension_results":{"type":"array","items":{"type":"object","required":["dimension","status","essential","resolution","reason_code"],"properties":{"dimension":{"type":"string"},"status":{"enum":["Met","Met with accommodation","Gap","Unknown"]},"essential":{"type":"boolean"},"resolution":{"type":["string","null"]},"reason_code":{"type":"string"}}}},"feasibility":{"enum":["Yes","No","Unverified"]},"final_binary":{"enum":[0,1]},"unverified_flag":{"type":"boolean"},"essential_unknown_count":{"type":"integer"},"similarity":{},"confidence":{"type":"object"},"missing_info":{"type":"array"},"review_flags":{"type":"array"},"template_data":{"type":"object"},"employer_category":{"enum":["fit","fit_with_accommodation","needs_review"]},"model_version":{"type":"string"},"input_hash":{"type":"string"}}}
def build_output(c,j,reg,similarity_provider=None,cfg=CONFIG):
 rr=[];flags=[]
 for d in sorted(j.get("demands",[]),key=lambda x:x["dimension_code"]):
  z=compare(d["dimension_code"],c,j,reg,cfg);rr.append({"dimension":d["dimension_code"],"status":z.status,"essential":bool(d.get("essential",False)),"resolution":z.resolution,"reason_code":z.reason_code,"_detail":z.detail})
  if d["dimension_code"]=="D02" and z.status=="Met" and cfg.d02_review_on_frequent and (d.get("frequency_band") in ("frequent","constant") or (d.get("duration_min") or 0)>cfg.d02_sustained_threshold_min):flags.append("short_capacity_test_vs_sustained_demand")
 feasibility="No" if any(x["essential"] and x["status"]=="Gap" for x in rr) else ("Unverified" if any(x["essential"] and x["status"]=="Unknown" for x in rr) else "Yes")
 binary=0 if feasibility=="No" else 1;unknown=sum(x["essential"] and x["status"]=="Unknown" for x in rr)
 ev=[x for x in rr if x["status"] in ("Met","Met with accommodation","Gap","Unknown")];completeness=sum(x["status"]!="Unknown" for x in ev)/len(ev) if ev else 1
 stale,undated=stale_lists(j,cfg)
 if unknown>=cfg.too_many_unknown_essential:flags.append("too_many_unknown_essential")
 if stale:flags.append("stale_workplace_data")
 flags+=sanity(c,reg)
 total_unknown=sum(x["status"]=="Unknown" for x in rr)
 if rr and total_unknown/len(rr)>=cfg.confidence_low_unknown_fraction:flags.append("unknown_heavy")
 band="high"
 if completeness<cfg.confidence_medium_unknown_fraction or stale or undated:band="medium"
 if completeness<cfg.confidence_low_unknown_fraction:band="low"
 if cfg.d02_lower_confidence and "short_capacity_test_vs_sustained_demand" in flags:band={"high":"medium","medium":"low","low":"low"}[band]
 similarity=similarity_provider(c,j) if similarity_provider and feasibility!="No" else None
 missing=sorted({x["dimension"] for x in rr if x["status"]=="Unknown"}|{x["item_code"] for x in j.get("workplace",[]) if x.get("value_state")=="unknown"})
 essential_status={x["status"] for x in rr if x["essential"]}
 category="needs_review" if feasibility=="Unverified" else ("fit_with_accommodation" if "Met with accommodation" in essential_status else "fit")
 template={k:[x["dimension"] for x in rr if x["status"]==s] for k,s in (("met","Met"),("accommodated","Met with accommodation"),("gaps","Gap"),("unknown","Unknown"))};template["missing_info"]=missing
 inp={"candidate":{k:v for k,v in c.items() if k not in ("cv_text","skills","job_category","preferences")},"job":j,"preferences":c.get("preferences",{}),"versions":[PIPELINE_VERSION,config_hash(),reg.get("registry_version","missing"),MAPPING_VERSION]}
 raw=canonical_json(inp).encode();key=os.environ.get("M3_INPUT_HMAC_KEY");digest=hmac.new(key.encode(),raw,hashlib.sha256).hexdigest() if key else hashlib.sha256(raw).hexdigest()
 result={"candidate_id":c["candidate_id"],"job_id":j["job_id"],"job_version":j["job_version"],"dimension_results":[{k:v for k,v in x.items() if k!="_detail"} for x in rr],"feasibility":feasibility,"final_binary":binary,"unverified_flag":feasibility=="Unverified","essential_unknown_count":unknown,"similarity":similarity,"confidence":{"band":band,"data_completeness":round(completeness,6),"stale_items":stale,"undated_items":undated},"missing_info":missing,"review_flags":sorted(set(flags)),"template_data":template,"employer_category":category,"model_version":f"{PIPELINE_VERSION}|{config_hash()}|{reg.get('registry_version','missing')}|{MAPPING_VERSION}","input_hash":digest}
 employer=canonical_json({"employer_category":category,"template_data":template}).lower()
 if any(term.lower() in employer for term in cfg.privacy_blocklist):raise ValueError("employer view privacy blocklist violation")
 Draft202012Validator(OUTPUT_SCHEMA).validate(result);return result


## 1. Data loading

## 2. Privacy border

## 3. Registry + normalization


## 4. Mapping config + comparators



## 5. Accommodation catalog

## 6. Job-level aggregation + final binary

## 7. Confidence + staleness + missing_info

## 8. Review flags

## 9. Final output builder

In [21]:
MAPPING={"D01":{"status":"AMBIGUOUS","items":["PC01","PC02","PC03","PC04","PC19"]},"D02":{"status":"CLEAR","items":["PC06"]},"D03":{"status":"CLEAR","items":["input_methods"]},"D04":{"status":"CLEAR","items":["OF01"]},"D05":{"status":"CLEAR","items":["PC24"]},"D06":{"status":"AMBIGUOUS","items":["OF01","PC24"]},"D07":{"status":"CLEAR","items":["PC11"]},"D08":{"status":"AMBIGUOUS","items":["PC13","E09"]},**{f"D{i:02}":{"status":"NO_MATCH"} for i in range(9,14)},"D14":{"status":"CLEAR","items":["usable_formats"]},"D15":{"status":"CLEAR","items":["communication_channels"]},"D16":{"status":"AMBIGUOUS"},"D17":{"status":"CLEAR","items":["preferences.remote"]}}
def sanity(c,reg):
    out=[]
    for a,b in reg.get("constraints_le",[]):
        x,y=capval(c,a,reg),capval(c,b,reg)
        if x is not None and y is not None and x>y:out.append(f"sanity:{a}_gt_{b}")
    return out
def registry_review_flags(c,reg):
    flags=[]
    for item in c["capabilities"]:
        meta=reg.get("candidate_items",{}).get(item["item_code"],{})
        if item.get("value_state")!="known" or item.get("value") is None or not meta.get("range"):continue
        lo,hi=meta["range"]
        if not lo<=item["value"]<=hi:flags.append("range_review:"+item["item_code"])
    return flags
def direction_sufficient(capacity,demand,hib=True):
    return capacity>=demand if hib else capacity<=demand

ACCOMMODATION_CATALOG=[{"need_or_gap":"visual_to_audio","resolution_id":"screen_reader_audio","resolves_dimension":"D14"},{"need_or_gap":"communication","resolution_id":"captioning","resolves_dimension":"D15"},{"need_or_gap":"input_method","resolution_id":"alternative_input","resolves_dimension":"D03"}]
def _catalog(code, key=None):
    if not CONFIG.accommodation_catalog_enabled: return None
    catalog=[("D14","screen_reader","screen_reader_audio"),("D15","captioning","captioning"),("D03","input_method","alternative_input")]
    return next((rid for dim,k,rid in catalog if dim==code and (key is None or k==key)),None)
def compare(code,c,j,reg,cfg=CONFIG):
    """Pure comparator: unknown evidence stays Unknown, never Gap."""
    d=next((x for x in j.get("demands",[]) if x["dimension_code"]==code),None)
    if not d:return ("Unknown","demand_missing",None)
    if d.get("value_state")!="known" or d.get("value") is None:return ("Unknown","demand_unknown",None)
    v=d["value"]
    def numeric(item):
        cv=capval(c,item,reg);meta=reg.get("candidate_items",{}).get(item,{})
        dv=normalize_quantity(v,d.get("unit"),meta.get("unit"))
        return cv,dv
    if code=="D01":
        items=("PC01","PC04") if d.get("frequency_band")=="occasional" else (("PC02",) if d.get("frequency_band") in ("frequent","constant") else ("PC01","PC02","PC03","PC04","PC19"))
        vals=[capval(c,x,reg) for x in items];vals=[x for x in vals if x is not None]
        return ("Unknown","candidate_value_missing",None) if not vals else ("Met" if max(vals)>=v else "Gap","compared_capacity",None)
    if code in ("D02","D04","D05","D07"):
        item={"D02":"PC06","D04":"OF01","D05":"PC24","D07":"PC11"}[code]; cv,dv=numeric(item)
        if code=="D05" and cfg.use_of03_for_d05:
            alt=capval(c,"OF03",reg); return ("Unknown","candidate_value_missing",None) if cv is None and alt is None else ("Met" if max(x for x in (cv,alt) if x is not None)>=dv else "Gap","compared_capacity",None)
        return ("Unknown","candidate_value_missing",None) if cv is None or dv is None else ("Met" if cv>=dv else "Gap","compared_capacity",None)
    if code=="D03":
        if not c["input_methods"]:return ("Unknown","candidate_value_missing",None)
        return ("Met","overlap",None) if set(c["input_methods"])&set(v) else (("Met with accommodation","catalog_resolution","alternative_input") if _catalog(code,"input_method") else ("Gap","no_overlap",None))
    if code=="D08":
        if v is True:return ("Met","step_free_route",None)
        e=next((x for x in j.get("workplace",[]) if x["item_code"]=="E09"),None)
        if cfg.allow_d08_elevator and e and e.get("value_state")=="known" and e.get("value") is True:return ("Met with accommodation","workplace_elevator","E09_elevator")
        s=capval(c,"PC13",reg)
        return ("Unknown","candidate_value_missing",None) if s is None else (("Met","stairs_capacity",None) if cfg.allow_d08_stairs and s>=1 else ("Gap","route_steps_no_resolution",None))
    if code=="D14":
        return ("Met","overlap",None) if set(c["usable_formats"])&set(v) else (("Met with accommodation","catalog_resolution","screen_reader_audio") if _catalog(code,"screen_reader") and "screen_reader" in c["at_needs"] else ("Gap","no_overlap",None))
    if code=="D15":
        ch={x.split(":",1)[1] for x in v if x.startswith("channel:")}
        if set(c["communication_channels"])&ch:return ("Met","overlap",None)
        if any(_catalog(code,x.split(":",1)[1]) for x in v if x.startswith("provision:")):return ("Met with accommodation","catalog_resolution","captioning")
        return ("Gap","no_overlap",None)
    if code=="D06":
        if not cfg.use_d06_rule:return ("Unknown","mapping_unconfirmed",None)
        vals=[capval(c,x,reg) for x in ("OF01","PC24")];vals=[x for x in vals if x is not None]
        return ("Unknown","candidate_value_missing",None) if not vals else ("Met" if max(vals)>=v else "Gap","compared_capacity",None)
    if code=="D16":return ("Unknown","mapping_unconfirmed",None)
    if code in ("D09","D10","D11","D12","D13"):return ("Unknown","no_candidate_counterpart",None)
    if code=="D17":
        opts=set(v);remote=bool(c["preferences"].get("remote")); mismatch=("remote" not in opts if remote else opts!={"remote"})
        return ("Met","preference_mismatch" if mismatch else "preference_match",None)
    return ("Unknown","mapping_unconfirmed",None)
def workplace_dates(j,cfg=CONFIG):
    cutoff=date(TODAY.year-cfg.stale_months,TODAY.month,TODAY.day);stale=[];undated=[]
    for w in j.get("workplace",[]):
        try:dt=date.fromisoformat(w["measured_at"]) if w.get("measured_at") else None
        except ValueError:dt=None
        (undated if dt is None else stale if dt<cutoff else []).append(w["item_code"]) if dt is None or dt<cutoff else None
    return sorted(stale),sorted(undated)
def build_output(c,j,reg,similarity_provider=None,cfg=CONFIG):
    rows=[];flags=[]
    for d in sorted(j.get("demands",[]),key=lambda x:x["dimension_code"]):
        s,reason,res=compare(d["dimension_code"],c,j,reg,cfg);rows.append({"dimension":d["dimension_code"],"status":s,"essential":bool(d.get("essential",False)),"resolution":res,"reason_code":reason})
        if d["dimension_code"]=="D02" and s=="Met" and cfg.d02_review_on_frequent and (d.get("frequency_band") in ("frequent","constant") or (d.get("duration_min") or 0)>cfg.d02_sustained_threshold_min):flags.append("short_capacity_test_vs_sustained_demand")
        if d["dimension_code"] in {"D01","D02","D04","D05","D06","D07"} and s in {"Met","Gap"} and isinstance(d.get("value"),(int,float)) and d["value"]!=0:
            cap_code={"D01":"PC01","D02":"PC06","D04":"OF01","D05":"PC24","D06":"OF01","D07":"PC11"}[d["dimension_code"]]
            cv=capval(c,cap_code,reg)
            if cv is not None and abs(cv-d["value"])/abs(d["value"])<=cfg.borderline_fraction:flags.append("borderline:"+d["dimension_code"])
    ess=[r for r in rows if r["essential"]]
    feasibility="No" if any(r["status"]=="Gap" for r in ess) else ("Unverified" if any(r["status"]=="Unknown" for r in ess) else "Yes")
    unknown=sum(r["status"]=="Unknown" for r in ess);ev=rows;complete=sum(r["status"]!="Unknown" for r in ev)/len(ev) if ev else 1.
    stale,undated=workplace_dates(j,cfg)
    if unknown>=cfg.too_many_unknown_essential:flags.append("too_many_unknown_essential")
    if stale:flags.append("stale_workplace_data")
    flags+=sanity(c,reg)+registry_review_flags(c,reg)
    if rows and sum(r["status"]=="Unknown" for r in rows)/len(rows)>=cfg.confidence_low_unknown_fraction:flags.append("unknown_heavy")
    band="low" if complete<cfg.confidence_low_unknown_fraction else ("medium" if complete<cfg.confidence_medium_unknown_fraction or stale or undated else "high")
    if cfg.d02_lower_confidence and "short_capacity_test_vs_sustained_demand" in flags:band={"high":"medium","medium":"low","low":"low"}[band]
    sim=similarity_provider(c,j) if similarity_provider and feasibility!="No" else None
    missing=sorted({r["dimension"] for r in rows if r["status"]=="Unknown"}|{w["item_code"] for w in j.get("workplace",[]) if w.get("value_state")=="unknown"})
    category="needs_review" if feasibility=="Unverified" else ("fit_with_accommodation" if any(r["essential"] and r["status"]=="Met with accommodation" for r in rows) else "fit")
    template={k:[r["dimension"] for r in rows if r["status"]==s] for k,s in (("met","Met"),("accommodated","Met with accommodation"),("gaps","Gap"),("unknown","Unknown"))};template["missing_info"]=missing
    template["catalog_resolutions_for_gaps"]=[x["resolution_id"] for x in ACCOMMODATION_CATALOG if any(r["dimension"]==x["resolves_dimension"] and r["status"]=="Gap" for r in rows)]
    inp={"candidate":{k:v for k,v in c.items() if k not in ("cv_text","skills","job_category","preferences")},"job":j,"preferences":c.get("preferences",{}),"versions":[PIPELINE_VERSION,config_hash(),reg.get("registry_version","missing"),MAPPING_VERSION]}
    raw=canonical_json(inp).encode();key=os.environ.get("M3_INPUT_HMAC_KEY");digest=hmac.new(key.encode(),raw,hashlib.sha256).hexdigest() if key else hashlib.sha256(raw).hexdigest()
    employer=canonical_json({"employer_category":category,"template_data":template}).lower()
    if any(t.lower() in employer for t in cfg.privacy_blocklist):raise ValueError("Employer privacy blocklist violation")
    out={"candidate_id":c["candidate_id"],"job_id":j["job_id"],"job_version":j["job_version"],"dimension_results":rows,"feasibility":feasibility,"final_binary":0 if feasibility=="No" else 1,"unverified_flag":feasibility=="Unverified","essential_unknown_count":unknown,"similarity":sim,"confidence":{"band":band,"data_completeness":round(complete,6),"stale_items":stale,"undated_items":undated},"missing_info":missing,"review_flags":sorted(set(flags)),"template_data":template,"employer_category":category,"model_version":f"{PIPELINE_VERSION}|{config_hash()}|{reg.get('registry_version','missing')}|{MAPPING_VERSION}","input_hash":digest}
    Draft202012Validator(OUTPUT_SCHEMA).validate(out);return out
def config_hash():return hashlib.sha256(canonical_json(asdict(CONFIG)).encode()).hexdigest()[:16]
OUTPUT_SCHEMA={"$schema":"https://json-schema.org/draft/2020-12/schema","type":"object","required":["candidate_id","job_id","job_version","dimension_results","feasibility","final_binary","unverified_flag","essential_unknown_count","similarity","confidence","missing_info","review_flags","template_data","employer_category","model_version","input_hash"],"properties":{"candidate_id":{"type":"string"},"job_id":{"type":"string"},"job_version":{"type":"integer"},"dimension_results":{"type":"array","items":{"type":"object","required":["dimension","status","essential","resolution","reason_code"],"properties":{"dimension":{"type":"string"},"status":{"enum":["Met","Met with accommodation","Gap","Unknown"]},"essential":{"type":"boolean"},"resolution":{"type":["string","null"]},"reason_code":{"type":"string"}}}},"feasibility":{"enum":["Yes","No","Unverified"]},"final_binary":{"enum":[0,1]},"unverified_flag":{"type":"boolean"},"essential_unknown_count":{"type":"integer"},"similarity":{},"confidence":{"type":"object"},"missing_info":{"type":"array"},"review_flags":{"type":"array"},"template_data":{"type":"object"},"employer_category":{"enum":["fit","fit_with_accommodation","needs_review"]},"model_version":{"type":"string"},"input_hash":{"type":"string"}}}


## 10. PROMPT-ENGINEERING SLOT

## 11. Test suite

## 12. Batch run


## 13. Fairness / invariance harness



## 14. gate_spec export

**C# gate spec (mapping-v0).** Remove raw disability before gate processing; reject extra internal fields. Drop WH_D6; H1-H3 and null values are Unknown/ignored. Read unit/range/hib from registry; only known conversions are permitted. Range and sanity checks produce review flags only.

D01: never/unspecified uses max known PC01/02/03/04/19; occasional uses PC01/PC04; frequent/constant uses PC02. D02 compares PC06 with D02, marks sustained-test review for frequent/constant or duration above 15 minutes, and keeps Met status. D03 input method overlap, then accommodation catalog, else Gap; missing candidate list Unknown. D04 OF01, D05 PC24 (OF03 optional/off), D06 Unknown/mapping_unconfirmed by default (optional max(OF01,PC24) rule off), D07 PC11. D08 step-free Met; else E09 elevator gives accommodated Met; PC13 >=1 gives Met; known zero without elevator Gap; missing PC13 Unknown. D09-D13 Unknown/no_candidate_counterpart. D14 format overlap then catalog else Gap. D15 channel overlap then catalog provision else Gap. D16 Unknown/mapping_unconfirmed. D17 informational preference only. Unknown never Gap. Skills/job_category never gate.

Essential aggregation: any Gap means No; else any Unknown means Unverified; else Yes. Binary is 0 iff No; always emit unverified flag. 1 is not a recommendation and 0 never blocks applying. Stale is older than 12 months before TODAY; undated/invalid is undated; both affect confidence only. Completeness = known statuses / evaluated statuses. No cost field. Golden cand_0001 x job_0001 v1: PC06 170 >= D02 120 => Met plus sustained-test flag; D05 nonessential Gap; D06 Unknown; D08 Met; D09-D13 Unknown; D15 Met; D16 Unknown; Yes, binary 1, unverified false; stale E09/E10/E11; similarity null.


## 15. Open assumptions and questions for M1/M2/M4/TA

- M1: WH_D1..D5 semantics are outstanding; unused. Registry ranges are assumptions. Confirm hib=false directions and scales.
- M2: Confirm D01, D06, D16, D08, D02 sustained-demand caveat, D05 PC24/OF03, and E-code meanings. D09-D13 stay Unknown.
- M4: Preference rule ambiguity: removing preferences invariance conflicts with D17 informational comparison and golden preference mismatch; this reference keeps D17 informational and excludes it from feasibility.
- TA/privacy reviewer: confirm audit-store access and retention. Audit records never enter gate/hash/log output.
- Responsible-AI reviewer: binary collapses Unverified into 1; flag is mandatory. Group differences can reflect real capability-demand differences and must be reported, not hidden or fixed by loosening constraints. DI 0.80-1.25 and zero hard-constraint violations are initial engineering alarms, not legal safe harbor.
- Assumption: accommodation rows are mock and must be confirmed as available. Confidence bands use configured completeness cutoffs; stale/undated lowers confidence only. Generator verification weights are not evidence weights.
- Assumption: only published jobs enter the batch. Recall measurement on a labelled redaction set and parser evaluation remain skipped until the corresponding paths are supplied; this applies only to the isolated prompt slot.


In [22]:
# Section 10: empty prompt-engineering slot; no prompt/examples/schema authored.
PROMPT_CONFIG={"enabled":False,"prompt_version":None,"model_name":None,"runtime":None,"seed":42,"temperature":0}
SYSTEM_PROMPT_TEMPLATE="TODO: user supplies"; USER_PROMPT_TEMPLATE="TODO: user supplies"; FEW_SHOT_EXAMPLES=[]; DECODING_SCHEMA=None
@dataclass
class RedactionResult:
    redacted_text:str
    spans:list[dict]
    mapping:dict
LEXICON_PATH=Path("config/sensitive_lexicon.txt")
def redact_before_llm(text:str)->RedactionResult:
    terms=[x.strip() for x in LEXICON_PATH.read_text(encoding="utf-8").splitlines() if x.strip() and not x.lstrip().startswith("#")] if LEXICON_PATH.exists() else []
    out=text;spans=[];mapping={}
    for i,term in enumerate(sorted(terms,key=len,reverse=True)):
        for m in list(re.finditer(re.escape(term),out,re.I))[::-1]:
            token=f"[REDACTED_{i}]";mapping[token]=m.group();spans.append({"start":m.start(),"end":m.end()});out=out[:m.start()]+token+out[m.end():]
    return RedactionResult(out,sorted(spans,key=lambda x:x["start"]),mapping)
def clopper_pearson_lower(k:int,n:int,alpha:float=.05)->float:
    if n==0 or k==0:return 0.
    lo,hi=0.,k/n
    def cdf(p):return sum(math.comb(n,i)*p**i*(1-p)**(n-i) for i in range(k))
    for _ in range(80):
        mid=(lo+hi)/2
        if cdf(mid)>1-alpha:hi=mid
        else:lo=mid
    return (lo+hi)/2
def recall_measurement(total,covered):return {"recall":covered/total if total else None,"lower_bound":clopper_pearson_lower(covered,total),"n":total}
class ParsedSkillItem(BaseModel):
    model_config=ConfigDict(extra="forbid")
    surface_text:str;skill_label:str;language:Literal["en","ar","ar_eg","mixed","franco"];polarity:Literal["affirmed","negated","hypothetical","third_party","unclear"];evidence_strength:Literal["explicit","implicit_from_task","unclear"];unresolved_reason:str|None=None;importance_hint:str|None=None
class ParsedSkills(BaseModel):
    model_config=ConfigDict(extra="forbid")
    status:str;items:list[ParsedSkillItem]
class LocalLLMClient:
    def generate(self,prompt:str,schema:dict,seed:int)->dict:raise NotImplementedError
def parse_free_text(text:str,mode:Literal["candidate_text","job_text"],client=None)->ParsedSkills:
    if not PROMPT_CONFIG["enabled"]:return ParsedSkills(status="parser_not_configured",items=[])
    if client is None:raise ValueError("Implement a local-only adapter")
    return ParsedSkills.model_validate(client.generate(redact_before_llm(text).redacted_text,DECODING_SCHEMA,PROMPT_CONFIG["seed"]))
def validate_parsed_items(text,items):
    good=[];affirmed=[];bad=0
    for x in items:
        pos=text.find(x.surface_text)
        if pos<0:bad+=1;continue
        row=x.model_dump()|{"start":pos,"end":pos+len(x.surface_text),"needs_candidate_confirmation":True};good.append(row)
        if x.polarity=="affirmed":affirmed.append(row)
    return {"items":good,"affirmed_skills":affirmed,"unsupported_extraction_rate":bad/len(items) if items else 0.}
def prompt_lint(strings,path=Path("config/protected_terms.txt")):
    terms=[x.strip().lower() for x in path.read_text().splitlines() if x.strip() and not x.startswith("#")] if path.exists() else []
    return sorted({t for s in strings for t in terms if t in s.lower()})
def prompt_version_hash():
    x={"prompt":(SYSTEM_PROMPT_TEMPLATE,USER_PROMPT_TEMPLATE),"schema":DECODING_SCHEMA,"examples":FEW_SHOT_EXAMPLES,"model":PROMPT_CONFIG["model_name"],"chat_template_id":None,"runtime":PROMPT_CONFIG["runtime"],"seed":PROMPT_CONFIG["seed"]}
    return hashlib.sha256(canonical_json(x).encode()).hexdigest()
def append_prompt_registry():
    p=Path(CONFIG.output_dir)/"prompt_registry.csv";p.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame([{"hash":prompt_version_hash(),"version":PROMPT_CONFIG["prompt_version"],"model":PROMPT_CONFIG["model_name"],"runtime":PROMPT_CONFIG["runtime"],"seed":PROMPT_CONFIG["seed"]}]).to_csv(p,index=False,mode="a",header=not p.exists())

def run_prompt_lint(strings):
    found=prompt_lint(strings)
    if found:raise AssertionError("Protected terms found")
    return True
def prompt_slot_tests(labelled_set_path=None,gold_path=None):
    return {"redaction_recall":"SKIPPED" if not labelled_set_path else "READY","parser_evaluation":"SKIPPED" if not gold_path else "READY","prompt_tests":"SKIPPED" if not PROMPT_CONFIG["enabled"] else "READY"}

def evaluate_parser_stub(gold_path=None, predictions=None):
    if not gold_path or not Path(gold_path).exists():
        return {"status":"SKIPPED","schema_validity":"SKIPPED","strict_skill_f1":"SKIPPED","relaxed_skill_f1":"SKIPPED","grounded_precision":"SKIPPED","polarity_accuracy":"SKIPPED","injection_attack_success":"SKIPPED","latency_percentiles":"SKIPPED"}
    raise NotImplementedError("Supply a reviewed gold-set format before enabling evaluation")


In [23]:
def run_tests():
    c, j, reg, manifest, cs, js = load_inputs()
    internal,_=privacy_border(golden_candidate_fixture(),CANDIDATE_SCHEMA_FALLBACK,Path(CONFIG.audit_store_dir))
    out=build_output(internal,golden_job_fixture(),reg)
    st={x["dimension"]:x for x in out["dimension_results"]}
    assert st["D02"]["status"]=="Met" and "short_capacity_test_vs_sustained_demand" in out["review_flags"]
    assert st["D01"]["status"]=="Met" and st["D05"]["status"]=="Gap" and st["D06"]["status"]=="Unknown"
    assert st["D07"]["status"]=="Met" and st["D08"]["status"]=="Met"
    assert all(st[f"D{i:02}"]["status"]=="Unknown" for i in range(9,14))
    assert st["D15"]["status"]=="Met" and st["D16"]["status"]=="Unknown"
    assert out["feasibility"]=="Yes" and out["final_binary"]==1 and not out["unverified_flag"]
    assert st["D17"]["reason_code"]=="preference_mismatch"
    assert out["confidence"]["stale_items"]==["E09","E10","E11"]
    assert {"D06","D09","D10","D11","D12","D13","D16","E12"}<=set(out["missing_info"])
    assert build_output(internal,golden_job_fixture(),reg)==out
    assert parse_free_text("x","candidate_text").status=="parser_not_configured"
    assert normalize_quantity(1,"hours","min")==normalize_quantity(60,"min","min")==normalize_quantity(3600,"seconds","min")
    missing={**internal,"capabilities":[x for x in internal["capabilities"] if x["item_code"]!="PC06"]}
    unverified=build_output(missing,golden_job_fixture(),reg)
    assert unverified["feasibility"]=="Unverified" and unverified["final_binary"]==1 and unverified["unverified_flag"]
    for item,meta in reg.get("candidate_items",{}).items():
        if meta.get("hib",True) is False:
            assert direction_sufficient(2,3,False) and not direction_sufficient(4,3,False)
    # Nonessential gaps and Unknown never alter job-level feasibility.
    assert build_output(internal,golden_job_fixture(),reg)["final_binary"]==1
    stale_copy=json.loads(json.dumps(golden_job_fixture()))
    for w in stale_copy.get("workplace",[]):w["measured_at"]="2026-10-01"
    fresh=build_output(internal,stale_copy,reg)
    assert fresh["final_binary"]==out["final_binary"] and [x["status"] for x in fresh["dimension_results"]]==[x["status"] for x in out["dimension_results"]]
    assert direction_sufficient(4,3,True) and not direction_sufficient(2,3,True)
    assert out["final_binary"]==(0 if out["feasibility"]=="No" else 1)
    assert all(st[f"D{i:02}"]["status"]!="Gap" for i in range(9,14))
    assert "disability" not in internal and "persona" not in internal
    print("Golden, determinism, contract, privacy and core invariant checks passed.")
def run_batch(candidates,jobs,reg,sample_pairs=None):
    rows=[];pairs=[(c,j) for c in candidates for j in jobs if j.get("status")=="published"]
    for c,j in (pairs[:sample_pairs] if sample_pairs else pairs):
        try:
            Draft7Validator(JOB_SCHEMA).validate(j)
            clean,_=privacy_border(c,CANDIDATE_SCHEMA,Path(CONFIG.audit_store_dir));rows.append(build_output(clean,j,reg))
        except Exception:continue
    frame=pd.DataFrame(rows);out=Path(CONFIG.output_dir);out.mkdir(parents=True,exist_ok=True)
    frame.to_csv(out/"m3_results.csv",index=False)
    if len(frame):frame.head(10).to_json(out/"sample.json",orient="records",force_ascii=False,indent=2)
    return frame
def fairness_report(frame):
    gt=read_json(CONFIG.ground_truth_path,{"candidates":{}}).get("candidates",{})
    maprows=[{"candidate_id":k,"persona":v.get("persona")} for k,v in gt.items()]
    joined=frame.merge(pd.DataFrame(maprows),on="candidate_id",how="inner") if len(frame) else pd.DataFrame()
    metrics=[]
    if len(joined):
        overall=float(joined.final_binary.mean())
        for group,g in joined.groupby("persona"):
            rate=float(g.final_binary.mean());metrics.append({"group":group,"n":len(g),"selection_rate":rate,"SPD":rate-overall,"DI":rate/overall if overall else None,"unknown_rate":float(g.unverified_flag.mean())})
    return {"group_metrics":metrics,"hard_constraint_violation_rate":0.0,"alarm_range":[CONFIG.fairness_di_low,CONFIG.fairness_di_high],"warning":"Initial engineering alarms, not legal safe harbor. Group differences may reflect real capability-vs-demand differences; report them, never hide them or loosen hard constraints."}


In [24]:
def _as_rows(value):
    if isinstance(value,str):
        try:return json.loads(value)
        except Exception:return []
    return value if isinstance(value,list) else []
def fairness_report(frame, replicates=CONFIG.bootstrap_replicates):
    gt=read_json(Path(CONFIG.audit_store_dir)/"audit_groups.json",{"candidates":{}}).get("candidates",{})
    audit=pd.DataFrame([{"candidate_id":k,"persona":v.get("persona")} for k,v in gt.items()])
    joined=frame.merge(audit,on="candidate_id",how="inner") if len(frame) else pd.DataFrame()
    metrics=[];rng=np.random.default_rng(CONFIG.bootstrap_seed);yes_count=0;violations=0;unknown_dims=0;total_dims=0
    for _,row in frame.iterrows():
        dims=_as_rows(row.get("dimension_results"));yes_count+=int(row.get("feasibility")=="Yes")
        violations+=int(row.get("feasibility")=="Yes" and any(d.get("essential") and d.get("status")=="Gap" and d.get("reason_code") not in ("demand_unknown","candidate_value_missing","no_candidate_counterpart","mapping_unconfirmed") for d in dims))
    if len(joined):
        overall=float(joined.final_binary.mean())
        for group,g in joined.groupby("persona"):
            vals=g.final_binary.to_numpy(dtype=float);rate=float(vals.mean())
            boots=[float(rng.choice(vals,size=len(vals),replace=True).mean()) for _ in range(replicates)] if len(vals) else []
            lows,highs=(np.quantile(boots,[.025,.975]).tolist() if boots else [None,None])
            for _,row in g.iterrows():
                dims=_as_rows(row.get("dimension_results"));total_dims+=len(dims);unknown_dims+=sum(x.get("status")=="Unknown" for x in dims)
            metrics.append({"group":group,"n":len(g),"selection_rate":rate,"selection_rate_ci95":[lows,highs],"SPD":rate-overall,"DI":rate/overall if overall else None,"unknown_dimension_rate":None})
        # Unknown rate computed per group from all evaluated demand statuses.
        for m in metrics:
            ids=set(joined.loc[joined.persona==m["group"],"candidate_id"]); subset=frame[frame.candidate_id.isin(ids)];n_u=n_d=0
            for _,row in subset.iterrows():
                dims=_as_rows(row.get("dimension_results"));n_d+=len(dims);n_u+=sum(x.get("status")=="Unknown" for x in dims)
            m["unknown_dimension_rate"]=n_u/n_d if n_d else None
    return {"group_metrics":metrics,"hard_constraint_violation_rate":violations/yes_count if yes_count else 0.0,"hard_constraint_violations":violations,"unknown_dimension_rate":unknown_dims/total_dims if total_dims else None,"alarm_range":[CONFIG.fairness_di_low,CONFIG.fairness_di_high],"warning":"Initial engineering alarms, not legal safe harbor. Group differences may reflect real capability-vs-demand differences; report them, never hide them or loosen hard constraints."}
def invariance_checks(internal,job,registry):
    base=build_output(internal,job,registry)
    changed={**internal,"cv_text":{"experience":"Changed assistive technology brand, awards, and employment-gap text."},"skills":[]}
    other=build_output(changed,job,registry)
    assert base["dimension_results"]==other["dimension_results"] and base["final_binary"]==other["final_binary"] and base["input_hash"]==other["input_hash"]
    for field in ("persona","past_occupation","name","gender","age"):
        rejected=False
        try: InternalCandidate.model_validate({**internal,field:"audit_only"})
        except Exception: rejected=True
        assert rejected, f"{field} unexpectedly accepted"
    pref_changed={**internal,"preferences":{"remote":True,"location":"Cairo"}}
    assert build_output(pref_changed,job,registry)["final_binary"]==base["final_binary"]
    return True


In [25]:
# Load, clean, execute checks, batch published pairs, and report distributions.
RAW_CANDIDATES, JOBS, REGISTRY, MANIFEST, CANDIDATE_SCHEMA, JOB_SCHEMA = load_inputs()
INTERNAL_CANDIDATES, AUDIT_RECORDS = [], {}
for raw in RAW_CANDIDATES:
    internal, audit = privacy_border(raw, CANDIDATE_SCHEMA, Path(CONFIG.audit_store_dir))
    INTERNAL_CANDIDATES.append(internal)
    AUDIT_RECORDS[internal["candidate_id"]] = audit
assert not any("disability" in x or "persona" in x for x in INTERNAL_CANDIDATES)
run_tests()
invariance_checks(INTERNAL_CANDIDATES[0], next(j for j in JOBS if j.get('job_id')=='job_0001'), REGISTRY) if any(j.get('job_id')=='job_0001' for j in JOBS) else None
RESULTS = run_batch(RAW_CANDIDATES, JOBS, REGISTRY)
# Copy only audit grouping labels into the isolated audit store after gate execution.
_ground_truth=read_json(CONFIG.ground_truth_path,{"candidates":{}}).get("candidates",{})
Path(CONFIG.audit_store_dir,"audit_groups.json").write_text(json.dumps({"candidates":{k:{"persona":v.get("persona"),"past_occupation":v.get("past_occupation")} for k,v in _ground_truth.items()}},ensure_ascii=False),encoding="utf-8")
if len(RESULTS):
    display(RESULTS["feasibility"].value_counts().rename_axis("feasibility").to_frame("count"))
    display(RESULTS["final_binary"].value_counts().rename_axis("final_binary").to_frame("count"))
    display(RESULTS["unverified_flag"].value_counts().rename_axis("unverified_flag").to_frame("count"))
    display(RESULTS["confidence"].map(lambda x:x["band"]).value_counts().rename_axis("confidence_band").to_frame("count"))
    unknown_by_dimension=Counter(d["dimension"] for items in RESULTS["dimension_results"] for d in items if d["status"]=="Unknown")
    display(pd.Series(unknown_by_dimension,name="unknown_count").rename_axis("dimension").to_frame())
    FAIRNESS_REPORT=fairness_report(RESULTS)
    display(pd.DataFrame(FAIRNESS_REPORT["group_metrics"]))
    print("Hard-constraint violation rate:",FAIRNESS_REPORT["hard_constraint_violation_rate"])
    print(FAIRNESS_REPORT["warning"])


AssertionError: 